# Phase 2B: Minimal Sequoia-Compliant Tree Speculation

**Goal.** Implement a minimal tree-structured speculative decoding method that applies Sequoia's three core ideas — DP-optimized topology, multi-child drafting, and tree-attention verification — scoped appropriately for T4. Benchmark against the same baseline/linear-SD/PLD results from Phase 1 on the same 10 prompts, same 30 trials methodology.

**Sequoia ideas implemented:**
1. **DP tree construction** (Sequoia Alg 4, Appendix F.1.1) with our Phase-2A-measured acceptance vector.
2. **Multi-child drafting** — at each parent, draft's top-b candidates populate b children (greedy analog of Sequoia's sampling-without-replacement).
3. **Tree-attention verification** — target processes all tree nodes in one forward pass with an ancestors-only attention mask; accept longest matching path (cover property in greedy limit).

**Sequoia ideas we do NOT implement (explicit scoping):**
- Sampling-without-replacement with residual updating (Sequoia Alg 2 blue lines) — equivalent to top-b under greedy, matters only under sampling.
- CUDA graph optimization — adds complexity without affecting the measurement we care about.
- Hardware-aware runtime tree selection — we run two fixed (n,d) configurations chosen by offline DP.

**What we will report.** For each configuration: measured speedup, DP-predicted speedup, and gap analysis. Two configurations: DP sweet spot (n=16, d=3 → predicted 1.68×) and a smaller sanity check (n=8, d=3 → predicted 1.31×).

In [1]:
!pip install -q transformers accelerate

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
import torch.nn.functional as F
import numpy as np
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Torch: 2.10.0+cu128, CUDA: True
GPU: Tesla T4


In [4]:
# ============================================================
# Upload swor_acceptance_vector.json from Phase 2A
# ============================================================
from google.colab import files
print("Upload swor_acceptance_vector.json from Phase 2A:")
uploaded = files.upload()

with open("swor_acceptance_vector.json") as f:
    phase2a = json.load(f)

ACC_VECTOR = phase2a["acceptance_vector"]  # p[1..5]
print(f"\nLoaded acceptance vector: p[1..5] = {[round(x, 4) for x in ACC_VECTOR]}")
print(f"Measured over {phase2a['n_samples']} samples")

Upload swor_acceptance_vector.json from Phase 2A:


Saving swor_acceptance_vector.json to swor_acceptance_vector.json

Loaded acceptance vector: p[1..5] = [0.8459, 0.5745, 0.55, 0.3333, 0.1667]
Measured over 305 samples


In [5]:
# ============================================================
# Load models (same pair as Phase 1 — comparability matters)
# ============================================================
TARGET_MODEL = "meta-llama/Llama-3.2-3B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda",
    attn_implementation="eager",  # we need custom attention masks
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="cuda",
    attn_implementation="eager",
)
draft_model.eval()

print(f"Target: {TARGET_MODEL} ({sum(p.numel() for p in target_model.parameters())/1e9:.2f}B)")
print(f"Draft : {DRAFT_MODEL} ({sum(p.numel() for p in draft_model.parameters())/1e9:.2f}B)")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target: meta-llama/Llama-3.2-3B (3.21B)
Draft : meta-llama/Llama-3.2-1B (1.24B)


In [6]:
# ============================================================
# Cell A: Sequoia DP (from paper Alg 4, Appendix F.1.1)
# ============================================================
# Solves for optimal tree of size <=N and depth <=D given acceptance
# vector p, under the positional acceptance assumption (Def 3.1).
# Returns (T, T_max, best_tree) where best_tree is the actual tree
# structure as a nested-list of children.

class TreeNode:
    __slots__ = ["children", "size"]
    def __init__(self, children=None):
        self.children = children or []
        self.size = 1 + sum(c.size for c in self.children)

    def flatten(self):
        """Return list of (parent_idx, depth) for each node in BFS order.
        Node 0 is the root (parent=-1, depth=0)."""
        out = [(-1, 0)]
        queue = [(self, 0)]  # (node, my_idx)
        while queue:
            node, my_idx = queue.pop(0)
            for child in node.children:
                child_idx = len(out)
                parent_depth = out[my_idx][1]
                out.append((my_idx, parent_depth + 1))
                queue.append((child, child_idx))
        return out

    def max_depth(self):
        if not self.children: return 0
        return 1 + max(c.max_depth() for c in self.children)


def sequoia_dp(acc_rates, max_size, max_depth, max_branch):
    """Build optimal tree structure via Sequoia DP.
    Returns the TreeNode root of the best tree found."""
    N, D, B = max_size, max_depth, max_branch
    P = np.asarray(acc_rates, dtype=float)
    if P.ndim == 1:
        P = np.tile(P, (D - 1, 1))
    if P.shape[1] < B + 1:
        P = np.pad(P, ((0, 0), (0, B + 1 - P.shape[1])))
    P = P[:D - 1, :B + 1]

    T = np.full((N + 1, D + 1, B + 1), -np.inf)
    T_max = np.full((N + 1, D + 1), -np.inf)
    T[1, 1:, 0] = 1.0
    T_max[1, 1:] = 1.0

    # best_new_node[n,d,b] = root of best subtree of size n, depth<=d as b-th child of some parent
    best_new_node = {(1, d, 0): None for d in range(1, D + 1)}
    best_tree = {(1, d): TreeNode() for d in range(1, D + 1)}

    for n in range(2, N + 1):
        for d in range(2, D + 1):
            for b in range(1, B + 1):
                vals = np.full(n - 1, -np.inf)
                for m in range(1, n):
                    prev = T[n - m, d, b - 1]
                    subtree = T_max[m, d - 1]
                    if np.isfinite(prev) and np.isfinite(subtree):
                        vals[m - 1] = prev + P[D - d, b] * subtree
                if np.any(np.isfinite(vals)):
                    T[n, d, b] = np.max(vals)
                    best_m = np.argmax(vals) + 1
                    best_new_node[n, d, b] = best_tree.get((best_m, d - 1))
            T_max[n, d] = np.max(T[n, d, :])

            if np.isfinite(T_max[n, d]):
                best_b = int(np.argmax(T[n, d, :]))
                kids = []
                remaining = n
                for b_idx in range(best_b, 0, -1):
                    next_child = best_new_node.get((remaining, d, b_idx))
                    if next_child is None:
                        break
                    kids.insert(0, next_child)
                    remaining -= next_child.size
                best_tree[n, d] = TreeNode(children=kids)

    return T_max, best_tree


# Run DP with measured acceptance vector
# Pad with small tail values for branches beyond what we measured
P_FULL = ACC_VECTOR + [0.05, 0.05, 0.02]  # k=6,7,8 — conservative tail

T_max, best_trees = sequoia_dp(P_FULL, max_size=64, max_depth=6, max_branch=5)

# Hardware-aware speedup formula from Sequoia §4.1
c_t4 = 0.596  # measured Cd/Ct
def t_n(n):
    if n <= 4: return 1.0
    if n <= 8: return 1.05
    if n <= 16: return 1.15
    if n <= 32: return 1.35
    return 1.70

def predicted_speedup(n, d):
    G = T_max[n, d]
    if not np.isfinite(G): return 0.0
    return G / (t_n(n) + d * c_t4)

print("DP-predicted speedups on T4 (with measured acceptance vector):")
print(f"{'n':>4} {'d':>3} {'G(n,d)':>8} {'Speedup':>8}")
for n, d in [(4, 2), (8, 3), (16, 3), (24, 3), (32, 3), (32, 4)]:
    G = T_max[n, d]
    s = predicted_speedup(n, d)
    print(f"{n:>4} {d:>3} {G:>8.3f} {s:>7.3f}x")

DP-predicted speedups on T4 (with measured acceptance vector):
   n   d   G(n,d)  Speedup
   4   2    2.458   1.121x
   8   3    3.722   1.312x
  16   3    4.941   1.682x
  24   3    5.371   1.712x
  32   3     -inf   0.000x
  32   4    7.234   1.937x


In [7]:
# ============================================================
# Cell B: Tree construction → flattened layout
# ============================================================
# Convert DP's tree structure into flat arrays usable on GPU:
#   parent[i]: parent node index (parent of root = -1)
#   depth[i]:  depth (root = 0)
#   position_ids[i]: prefix_len + depth[i]  (for RoPE positional encoding)
#   attn_mask[i, j]: True iff i can attend to j (j is ancestor of i or == i)

def build_tree_layout(tree_root):
    """Returns (parent, depth, size)."""
    layout = tree_root.flatten()  # [(parent_idx, depth), ...]
    parent = np.array([p for p, _ in layout], dtype=np.int64)
    depth  = np.array([d for _, d in layout], dtype=np.int64)
    return parent, depth

def build_tree_attn_mask(parent, device):
    """Ancestors-only attention mask. Returns (n, n) bool where mask[i,j]=True
    iff j is an ancestor of i (or j == i)."""
    n = len(parent)
    mask = torch.zeros((n, n), dtype=torch.bool, device=device)
    for i in range(n):
        j = i
        while j != -1:
            mask[i, j] = True
            j = int(parent[j])
    return mask

def tree_summary(tree_root):
    parent, depth = build_tree_layout(tree_root)
    children_of = {i: [] for i in range(len(parent))}
    for i, p in enumerate(parent):
        if p >= 0: children_of[int(p)].append(i)
    branching = [len(children_of[i]) for i in range(len(parent)) if children_of[i]]
    return {
        "size": len(parent),
        "depth": int(depth.max()),
        "avg_branching": float(np.mean(branching)) if branching else 0.0,
        "max_branching": max(branching) if branching else 0,
    }

# Select our benchmark configurations
TREE_CONFIGS = [(8, 3), (16, 3)]
BENCHMARK_TREES = {}
for n, d in TREE_CONFIGS:
    tree = best_trees.get((n, d))
    if tree is None:
        print(f"WARNING: no tree found for (n={n}, d={d})")
        continue
    BENCHMARK_TREES[(n, d)] = tree
    summary = tree_summary(tree)
    print(f"Tree (n={n}, d={d}): {summary}")
    print(f"  predicted speedup = {predicted_speedup(n, d):.3f}x")

# Choose PRIMARY and SECONDARY config
PRIMARY = (16, 3)
SECONDARY = (8, 3)
print(f"\nPrimary benchmark: (n={PRIMARY[0]}, d={PRIMARY[1]})")
print(f"Secondary benchmark: (n={SECONDARY[0]}, d={SECONDARY[1]})")

Tree (n=8, d=3): {'size': 8, 'depth': 2, 'avg_branching': 2.3333333333333335, 'max_branching': 3}
  predicted speedup = 1.312x
Tree (n=16, d=3): {'size': 16, 'depth': 2, 'avg_branching': 3.0, 'max_branching': 4}
  predicted speedup = 1.682x

Primary benchmark: (n=16, d=3)
Secondary benchmark: (n=8, d=3)


In [8]:
# ============================================================
# Cell C (FIXED): Tree speculation runtime with incremental draft KV cache
# ============================================================
# FIX RATIONALE (vs. original):
#   The original draft_populate_tree reprocessed the full prefix for EVERY
#   interior tree node (one draft forward per parent, each ingesting prefix +
#   path). For n=16,d=3 with ~5-7 interior nodes and prefix ~30-150 tokens,
#   this meant draft cost per iteration was dominated by repeated prefix
#   reprocessing -- not the algorithm's intended O(d) incremental cost.
#
#   Sequoia's speedup formula S = G(n,d) / (t(n) + d*c) assumes draft does
#   incremental decoding with a persistent KV cache. We now do that.
#
# STRATEGY:
#   1. One full draft forward on the prefix -> (prefix_kv, root_logits).
#      Root logits determine depth-1 children. Node 0 is a virtual root
#      (no token of its own; its "token" is implicit in the prefix).
#   2. For each interior non-root node (depth>=1 with children), we extend
#      its PARENT's KV cache by exactly ONE token (this node's tree_token) to
#      get its own logits, which pick its children. O(1 token) per node.
#   3. When a parent has multiple interior children, fork (deep-copy) the
#      parent's cache so each subtree extends an independent branch. The
#      LAST interior child consumes the cache in place (no fork).
#
# Cost accounting for (n=16, d=3):
#   - 1 prefix forward (prefix_len tokens).
#   - ~(interior non-root nodes) single-token forwards. Typical 3-5.
#   - ~(interior non-root nodes at depth < max_depth) cache forks, each
#     a few MiB of tensor clones. Trivial vs. a full-prefix forward.

import time
import copy
from transformers import DynamicCache


def _fork_cache(cache):
    """Return an independent deep copy of a Cache object.

    Version-safe across transformers refactors of DynamicCache: uses
    copy.deepcopy, which recurses into whatever internal layout the cache
    exposes (old .key_cache/.value_cache lists, new .layers[i].keys/.values,
    or anything in between). torch.Tensor.__deepcopy__ clones storage, so
    the forked cache can be mutated without affecting the source.
    """
    return copy.deepcopy(cache)


def _cache_seq_len(cache) -> int:
    """Sequence length of a Cache (public API, stable across versions)."""
    return int(cache.get_seq_length())


# Global instrumentation (reset at benchmark start, read after).
# These let us attribute speedup change to the draft cost reduction in the report.
_PROF = {
    "draft_prefix_ms_sum": 0.0,
    "draft_prefix_calls": 0,
    "draft_extend_ms_sum": 0.0,
    "draft_extend_calls": 0,
    "verify_ms_sum": 0.0,
    "verify_calls": 0,
    "iter_count": 0,
}

def _prof_reset():
    for k in list(_PROF.keys()):
        _PROF[k] = 0 if isinstance(_PROF[k], int) else 0.0


@torch.inference_mode()
def draft_populate_tree(prefix_ids, tree_parent, tree_depth, max_branch):
    """Populate tree_tokens using incremental draft forwards with per-branch caches.
    Returns tree_tokens: tensor of shape (n_nodes,) with token IDs.
    """
    n_nodes = len(tree_parent)
    device = prefix_ids.device
    tree_tokens = torch.zeros(n_nodes, dtype=torch.long, device=device)

    # children-of map
    children_of = {i: [] for i in range(n_nodes)}
    for i, p in enumerate(tree_parent.tolist()):
        if int(p) >= 0:
            children_of[int(p)].append(i)

    def _is_interior(i):
        return len(children_of[i]) > 0

    interior_children_count = {
        i: sum(1 for c in children_of[i] if _is_interior(c))
        for i in range(n_nodes)
    }

    # --- Step 1: One full draft forward on the prefix ---
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    prefix_cache = DynamicCache()
    draft_out = draft_model(
        input_ids=prefix_ids,
        past_key_values=prefix_cache,
        use_cache=True,
    )
    prefix_cache = draft_out.past_key_values
    root_logits = draft_out.logits[0, -1, :]
    torch.cuda.synchronize()
    _PROF["draft_prefix_ms_sum"] += (time.perf_counter() - t0) * 1000.0
    _PROF["draft_prefix_calls"] += 1

    node_cache = {0: prefix_cache}
    remaining = dict(interior_children_count)

    # Populate depth-1 child tokens from root logits
    root_kids = children_of[0]
    if root_kids:
        topk = torch.topk(root_logits, k=len(root_kids)).indices
        for c_idx, child in enumerate(root_kids):
            tree_tokens[child] = topk[c_idx]

    # --- BFS over interior non-root nodes (they need own logits for their children) ---
    max_depth = int(tree_depth.max()) if n_nodes > 0 else 0
    for dep in range(1, max_depth):
        nodes_at_depth = [
            i for i in range(n_nodes)
            if int(tree_depth[i]) == dep and _is_interior(i)
        ]
        for node in nodes_at_depth:
            parent = int(tree_parent[node])

            pcache_src = node_cache[parent]
            remaining[parent] -= 1
            if remaining[parent] > 0:
                pcache = _fork_cache(pcache_src)
            else:
                pcache = pcache_src
                # Drop reference (except for root to keep logic simple)
                if parent != 0:
                    node_cache.pop(parent, None)

            # Extend by one token
            this_tok = tree_tokens[node].view(1, 1)
            cache_pos = _cache_seq_len(pcache)
            cache_position = torch.arange(
                cache_pos, cache_pos + 1, device=device
            )

            torch.cuda.synchronize()
            t1 = time.perf_counter()
            d_out = draft_model(
                input_ids=this_tok,
                past_key_values=pcache,
                use_cache=True,
                cache_position=cache_position,
            )
            torch.cuda.synchronize()
            _PROF["draft_extend_ms_sum"] += (time.perf_counter() - t1) * 1000.0
            _PROF["draft_extend_calls"] += 1

            node_cache[node] = d_out.past_key_values
            node_logits = d_out.logits[0, -1, :]

            kids = children_of[node]
            topk = torch.topk(node_logits, k=len(kids)).indices
            for c_idx, child in enumerate(kids):
                tree_tokens[child] = topk[c_idx]

    del node_cache
    return tree_tokens


# ==========  verify_tree and accept_longest_path: UNCHANGED from original ==========
# They operate correctly on the tree_tokens produced above.

@torch.inference_mode()
def verify_tree(prefix_ids, tree_tokens, tree_parent, tree_depth):
    """Run target model on prefix + tree_tokens with tree attention mask.
    Returns target's argmax prediction at each tree node (shape: n_nodes,)."""
    n_nodes = len(tree_tokens)
    device = prefix_ids.device
    prefix_len = prefix_ids.shape[1]

    # Re-pack: use only nodes with depth >= 1
    real_mask = tree_depth >= 1
    real_idx = torch.where(real_mask)[0].tolist()
    real_tokens = tree_tokens[real_idx]
    n_real = len(real_idx)

    if n_real == 0:
        t_out = target_model(prefix_ids)
        return None

    full_input_ids = torch.cat([prefix_ids, real_tokens.unsqueeze(0)], dim=1)
    total_len = full_input_ids.shape[1]
    mask_2d = torch.zeros(total_len, total_len, dtype=torch.bool, device=device)
    for i in range(prefix_len):
        mask_2d[i, :i + 1] = True

    old_to_new = {old: new for new, old in enumerate(real_idx)}

    for new_i, old_i in enumerate(real_idx):
        row = prefix_len + new_i
        mask_2d[row, :prefix_len] = True
        j = old_i
        while j != -1:
            if j in old_to_new:
                mask_2d[row, prefix_len + old_to_new[j]] = True
            j = int(tree_parent[j])

    attn_mask_4d = torch.where(
        mask_2d.unsqueeze(0).unsqueeze(0),
        torch.tensor(0.0, dtype=torch.float16, device=device),
        torch.tensor(float("-inf"), dtype=torch.float16, device=device),
    )

    real_depths = tree_depth[real_idx].tolist()
    tree_pos = torch.tensor(
        [prefix_len - 1 + int(d) for d in real_depths], device=device
    )
    prefix_pos = torch.arange(prefix_len, device=device)
    position_ids = torch.cat([prefix_pos, tree_pos]).unsqueeze(0)

    t_out = target_model(
        input_ids=full_input_ids,
        attention_mask=attn_mask_4d,
        position_ids=position_ids,
    )

    target_preds_for_node = torch.zeros(n_real, dtype=torch.long, device=device)
    for new_i, old_i in enumerate(real_idx):
        parent_old = int(tree_parent[old_i])
        if parent_old == 0:
            parent_pos_in_packed = prefix_len - 1
        else:
            parent_new = old_to_new[parent_old]
            parent_pos_in_packed = prefix_len + parent_new
        target_preds_for_node[new_i] = t_out.logits[0, parent_pos_in_packed, :].argmax()

    bonus_preds = {new_i: t_out.logits[0, prefix_len + new_i, :].argmax().item()
                   for new_i in range(n_real)}

    return {
        "real_idx": real_idx,
        "old_to_new": old_to_new,
        "target_preds": target_preds_for_node,
        "bonus_preds": bonus_preds,
    }


def accept_longest_path(tree_tokens, tree_parent, tree_depth, verify_result):
    """Walk from virtual root, descend greedily along the longest path where
    each child's token == target's prediction (cover property)."""
    if verify_result is None:
        return [], None
    real_idx = verify_result["real_idx"]
    old_to_new = verify_result["old_to_new"]
    target_preds = verify_result["target_preds"].tolist()
    bonus_preds = verify_result["bonus_preds"]

    children_of = {-1: [], 0: []}
    for i in range(len(tree_parent)):
        children_of[i] = []
    for i, p in enumerate(tree_parent.tolist()):
        children_of[int(p)].append(i)

    accepted = []
    current = 0
    last_real_new = None
    while True:
        kids = children_of.get(current, [])
        accepted_kid = None
        for kid in kids:
            if kid not in old_to_new: continue
            new_i = old_to_new[kid]
            if int(tree_tokens[kid]) == target_preds[new_i]:
                accepted_kid = kid
                accepted.append(int(tree_tokens[kid]))
                last_real_new = new_i
                break
        if accepted_kid is None:
            break
        current = accepted_kid

    if last_real_new is not None:
        bonus = bonus_preds[last_real_new]
    else:
        bonus = None

    return accepted, bonus


@torch.inference_mode()
def tree_spec_decode(prompt, tree_root, max_new_tokens=128):
    """Top-level: run tree-structured speculative decoding end-to-end."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    prefix_len_orig = input_ids.shape[1]

    parent_arr, depth_arr = build_tree_layout(tree_root)
    tree_parent = torch.tensor(parent_arr, device="cuda")
    tree_depth = torch.tensor(depth_arr, device="cuda")

    n_iter = 0
    total_accepted_drafts = 0

    while (input_ids.shape[1] - prefix_len_orig) < max_new_tokens:
        tree_tokens = draft_populate_tree(input_ids, tree_parent, tree_depth, max_branch=5)

        torch.cuda.synchronize()
        t_v = time.perf_counter()
        verify_res = verify_tree(input_ids, tree_tokens, tree_parent, tree_depth)
        torch.cuda.synchronize()
        _PROF["verify_ms_sum"] += (time.perf_counter() - t_v) * 1000.0
        _PROF["verify_calls"] += 1

        accepted, bonus = accept_longest_path(tree_tokens, tree_parent, tree_depth, verify_res)

        new_tokens = accepted.copy()
        if bonus is not None:
            new_tokens.append(bonus)
        else:
            t_out = target_model(input_ids)
            new_tokens.append(int(t_out.logits[0, -1, :].argmax()))

        if not new_tokens:
            break
        new_tensor = torch.tensor([new_tokens], device="cuda")
        input_ids = torch.cat([input_ids, new_tensor], dim=1)
        n_iter += 1
        _PROF["iter_count"] += 1
        total_accepted_drafts += len(accepted)

        if tokenizer.eos_token_id in new_tokens:
            break

    return input_ids, n_iter, total_accepted_drafts


print("Tree spec decode runtime defined (KV-cache-aware draft).")


Tree spec decode runtime defined (KV-cache-aware draft).


In [9]:
# ============================================================
# Cell D: Correctness checks BEFORE benchmarking
# ============================================================

print("=" * 60)
print("CORRECTNESS CHECK 1: greedy baseline output")
print("=" * 60)
test_prompt = "The history of artificial intelligence began in the 1950s when researchers first"
test_input = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    baseline_out = target_model.generate(
        **test_input, max_new_tokens=32,
        do_sample=False, pad_token_id=tokenizer.pad_token_id,
    )
baseline_ids = baseline_out[0].tolist()
baseline_text = tokenizer.decode(baseline_ids[test_input['input_ids'].shape[1]:])
print(f"Baseline continuation: {baseline_text[:100]}...")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CORRECTNESS CHECK 1: greedy baseline output
Baseline continuation:  began to explore the idea of creating machines that could think and learn like humans. Since then, ...


In [10]:
# ============================================================
# Correctness check 2: tree output matches baseline (greedy)
# ============================================================
print("=" * 60)
print("CORRECTNESS CHECK 2: tree output matches greedy baseline")
print("=" * 60)

tree_primary = BENCHMARK_TREES[PRIMARY]
tree_ids, n_iter, n_accept = tree_spec_decode(test_prompt, tree_primary, max_new_tokens=32)
tree_text = tokenizer.decode(tree_ids[0, test_input['input_ids'].shape[1]:].tolist())

# Compare first 20 tokens
prefix_len = test_input['input_ids'].shape[1]
baseline_gen = baseline_ids[prefix_len:prefix_len + 20]
tree_gen = tree_ids[0, prefix_len:prefix_len + 20].tolist()

n_match = sum(1 for a, b in zip(baseline_gen, tree_gen) if a == b)
print(f"Baseline first 20: {baseline_gen}")
print(f"Tree first 20:     {tree_gen}")
print(f"Matching tokens: {n_match}/20")
print(f"Tree iterations: {n_iter}, total draft tokens accepted: {n_accept}")
print(f"\nTree text: {tree_text[:100]}...")

if n_match >= 18:
    print("\n✓ Tree output substantially matches baseline (expected for greedy)")
else:
    print(f"\n⚠ WARNING: only {n_match}/20 tokens match — possible bug in tree implementation")

CORRECTNESS CHECK 2: tree output matches greedy baseline
Baseline first 20: [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
Tree first 20:     [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
Matching tokens: 20/20
Tree iterations: 12, total draft tokens accepted: 22

Tree text:  began to explore the idea of creating machines that could think and learn like humans. Since then, ...

✓ Tree output substantially matches baseline (expected for greedy)


In [11]:
# ============================================================
# Cell D2 (NEW): Per-call incremental decode cost probe (Cd, Ct, fork)
# ============================================================
# Purpose: close the loop on the framework-amortization claim.
# Checkpoint 2 assumed in-pipeline Cd is ~10x smaller than standalone Cd
# because of KV-cache persistence. Our Phase 2B profiler shows extend-per-call
# cost of ~22ms -- identical to standalone Cd. This cell isolates the question:
# is per-call incremental Cd truly ~constant across cache lengths, which would
# prove the bandwidth-floor hypothesis?
#
# What we measure (each: 5 warmup + 50 timed trials, CUDA-synced):
#   - Draft 1-token extend from cache of length L for L in {30, 64, 128, 256}
#   - Target 1-token extend from cache of length L for the same L
#   - copy.deepcopy(cache) for L=128 (the fork cost itself)
#   - Draft and Target standalone Cd/Ct via tiny autoregressive loop (sanity check
#     against Phase 1c's 22.42 / 37.63 ms/token)
#
# Expected signature of bandwidth-bound regime:
#   - Cd_extend(L) approximately flat across L, at ~22 ms for 1B
#   - Ct_extend(L) approximately flat across L, at ~37 ms for 3B
#   - Fork cost much smaller than either (a few ms at most)

import time
import copy
from transformers import DynamicCache

print("=" * 70)
print("Cd/Ct PROBE: per-call incremental decode cost as a function of cache length")
print("=" * 70)


def _measure_extend(model, cache_len, n_trials=50, n_warmup=5):
    """Build a cache of length cache_len by running the model once on a random
    prefix, then time n_trials single-token extends with full CUDA syncs."""
    # Build a realistic prefix and populate the cache once
    prefix = torch.randint(100, 30000, (1, cache_len), device="cuda")
    cache = DynamicCache()
    with torch.inference_mode():
        out = model(input_ids=prefix, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    actual_len = int(cache.get_seq_length())
    assert actual_len == cache_len, f"expected cache len {cache_len}, got {actual_len}"

    # Pre-compute the extend token and cache_position (constants across trials)
    new_tok = torch.tensor([[1234]], device="cuda")

    # Warmup
    with torch.inference_mode():
        for _ in range(n_warmup):
            work_cache = copy.deepcopy(cache)
            cp = torch.arange(cache_len, cache_len + 1, device="cuda")
            model(input_ids=new_tok, past_key_values=work_cache,
                  use_cache=True, cache_position=cp)

    # Timed trials: deep-copy cache each time so we measure extend-from-fixed-length,
    # NOT the (deepening) extend from a growing cache.
    latencies = []
    with torch.inference_mode():
        for _ in range(n_trials):
            work_cache = copy.deepcopy(cache)
            cp = torch.arange(cache_len, cache_len + 1, device="cuda")
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(input_ids=new_tok, past_key_values=work_cache,
                  use_cache=True, cache_position=cp)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000.0)
    return {
        "cache_len": cache_len,
        "mean_ms": float(np.mean(latencies)),
        "median_ms": float(np.median(latencies)),
        "std_ms": float(np.std(latencies)),
        "p95_ms": float(np.percentile(latencies, 95)),
    }


def _measure_fork_cost(model, cache_len, n_trials=50, n_warmup=5):
    """Time copy.deepcopy on a populated cache."""
    prefix = torch.randint(100, 30000, (1, cache_len), device="cuda")
    cache = DynamicCache()
    with torch.inference_mode():
        out = model(input_ids=prefix, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    for _ in range(n_warmup):
        _ = copy.deepcopy(cache)
    latencies = []
    for _ in range(n_trials):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = copy.deepcopy(cache)
        torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000.0)
    return {
        "cache_len": cache_len,
        "mean_ms": float(np.mean(latencies)),
        "median_ms": float(np.median(latencies)),
    }


def _measure_standalone_ar(model, n_new=20, n_warmup=3, n_trials=10):
    """Standalone autoregressive: start from a tiny prompt, decode n_new tokens
    WITH persistent KV cache in a manual loop. Reports mean ms/token.
    This is the apples-to-apples analog of Phase 1c standalone Cd/Ct."""
    prefix = torch.randint(100, 30000, (1, 8), device="cuda")
    # Warmup
    with torch.inference_mode():
        for _ in range(n_warmup):
            cache = DynamicCache()
            out = model(input_ids=prefix, past_key_values=cache, use_cache=True)
            tok = out.logits[0, -1].argmax().view(1, 1)
            for step in range(n_new):
                cp = torch.arange(
                    int(out.past_key_values.get_seq_length()),
                    int(out.past_key_values.get_seq_length()) + 1,
                    device="cuda",
                )
                out = model(input_ids=tok, past_key_values=out.past_key_values,
                            use_cache=True, cache_position=cp)
                tok = out.logits[0, -1].argmax().view(1, 1)

    latencies = []
    with torch.inference_mode():
        for _ in range(n_trials):
            cache = DynamicCache()
            out = model(input_ids=prefix, past_key_values=cache, use_cache=True)
            tok = out.logits[0, -1].argmax().view(1, 1)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for step in range(n_new):
                cp = torch.arange(
                    int(out.past_key_values.get_seq_length()),
                    int(out.past_key_values.get_seq_length()) + 1,
                    device="cuda",
                )
                out = model(input_ids=tok, past_key_values=out.past_key_values,
                            use_cache=True, cache_position=cp)
                tok = out.logits[0, -1].argmax().view(1, 1)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000.0 / n_new)
    return {
        "ms_per_token": float(np.mean(latencies)),
        "std_ms_per_token": float(np.std(latencies)),
    }


PROBE_LENS = [30, 64, 128, 256]
probe_results = {"draft_extend": {}, "target_extend": {},
                 "fork_cost": {}, "standalone_ar": {}}

print("\n--- Draft (1B) 1-token extend cost vs. cache length ---")
print(f"{'L':>6} {'mean ms':>10} {'median ms':>10} {'std ms':>8} {'p95 ms':>8}")
for L in PROBE_LENS:
    r = _measure_extend(draft_model, L)
    probe_results["draft_extend"][L] = r
    print(f"{L:>6} {r['mean_ms']:>10.2f} {r['median_ms']:>10.2f} {r['std_ms']:>8.2f} {r['p95_ms']:>8.2f}")

print("\n--- Target (3B) 1-token extend cost vs. cache length ---")
print(f"{'L':>6} {'mean ms':>10} {'median ms':>10} {'std ms':>8} {'p95 ms':>8}")
for L in PROBE_LENS:
    r = _measure_extend(target_model, L)
    probe_results["target_extend"][L] = r
    print(f"{L:>6} {r['mean_ms']:>10.2f} {r['median_ms']:>10.2f} {r['std_ms']:>8.2f} {r['p95_ms']:>8.2f}")

print("\n--- Fork cost (copy.deepcopy of cache) ---")
print(f"{'L':>6} {'draft mean ms':>16} {'target mean ms':>18}")
for L in PROBE_LENS:
    rd = _measure_fork_cost(draft_model, L, n_trials=30, n_warmup=3)
    rt = _measure_fork_cost(target_model, L, n_trials=30, n_warmup=3)
    probe_results["fork_cost"][L] = {"draft": rd, "target": rt}
    print(f"{L:>6} {rd['mean_ms']:>16.2f} {rt['mean_ms']:>18.2f}")

print("\n--- Standalone autoregressive (loop with persistent cache) ---")
r_d = _measure_standalone_ar(draft_model)
r_t = _measure_standalone_ar(target_model)
probe_results["standalone_ar"]["draft"] = r_d
probe_results["standalone_ar"]["target"] = r_t
print(f"Draft  1B: {r_d['ms_per_token']:.2f} +/- {r_d['std_ms_per_token']:.2f} ms/token  "
      f"(Phase 1c standalone: 22.42 ms/token)")
print(f"Target 3B: {r_t['ms_per_token']:.2f} +/- {r_t['std_ms_per_token']:.2f} ms/token  "
      f"(Phase 1c standalone: 37.63 ms/token)")

# Derive the effective cost ratio from this probe
c_probe = r_d['ms_per_token'] / r_t['ms_per_token']
print(f"\nc = Cd/Ct (from probe) = {c_probe:.3f}")
print(f"c = Cd/Ct (from Phase 1c) = 0.596")

print("\n" + "=" * 70)
print("KEY FINDING CHECK:")
print("=" * 70)
d_extends = [probe_results['draft_extend'][L]['mean_ms'] for L in PROBE_LENS]
d_range = max(d_extends) - min(d_extends)
d_mean = sum(d_extends) / len(d_extends)
print(f"Draft extend cost range: {min(d_extends):.1f} .. {max(d_extends):.1f} ms "
      f"(spread {d_range:.1f} ms, mean {d_mean:.1f} ms)")
if d_range / d_mean < 0.20:
    print(f"  -> Draft Cd_extend is ~flat across L ({d_range/d_mean*100:.1f}% spread) "
          "→ BANDWIDTH-BOUND regime confirmed.")
else:
    print(f"  -> Draft Cd_extend varies significantly with L ({d_range/d_mean*100:.1f}% spread) "
          "→ KV attention cost matters at these lengths.")

# Save probe results alongside the main output
with open("cd_probe_results.json", "w") as f:
    json.dump(probe_results, f, indent=2)
print("\nSaved cd_probe_results.json")


Cd/Ct PROBE: per-call incremental decode cost as a function of cache length

--- Draft (1B) 1-token extend cost vs. cache length ---
     L    mean ms  median ms   std ms   p95 ms
    30      20.06      19.56     1.52    22.98
    64      20.22      19.53     1.90    25.05
   128      20.08      19.55     1.83    22.02
   256      19.77      19.34     1.41    22.02

--- Target (3B) 1-token extend cost vs. cache length ---
     L    mean ms  median ms   std ms   p95 ms
    30      43.49      43.53     6.69    55.58
    64      35.86      34.36     4.79    47.61
   128      34.90      34.07     2.90    41.76
   256      35.55      34.55     2.89    41.27

--- Fork cost (copy.deepcopy of cache) ---
     L    draft mean ms     target mean ms
    30             2.36               4.17
    64             2.37               5.29
   128             2.54               4.80
   256             2.33               4.11

--- Standalone autoregressive (loop with persistent cache) ---
Draft  1B: 23.34

In [12]:
# ============================================================
# Cell E (updated): Benchmark harness with draft/verify profiling
# ============================================================
PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "In computer science, a hash table is a data structure that implements an associative",
    "The process of photosynthesis converts light energy into chemical energy through",
    'def fibonacci(n):\n    """Calculate the nth Fibonacci number."""\n    if n <= 1:\n        return n\n',
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
    "Summarize the following: Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Machine learning algorithms use historical data as input to predict new output values. Machine learning is",
    "Write a detailed explanation of how TCP/IP networking works, starting from",
    "Explain the difference between a stack and a queue data structure. A stack",
    "The transformer architecture was introduced in the paper Attention is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The key innovation was the multi-head attention mechanism, which allows the model to attend to different parts of the input simultaneously. Since then, transformers have become the foundation for",
    "Large language models are trained on massive datasets of text from the internet. During training, the model learns to predict the next token in a sequence given all previous tokens. This autoregressive training objective means that at inference time, the model generates text one token at a time, each conditioned on all previously generated tokens. This sequential nature creates a fundamental bottleneck because",
]
PROMPT_LABELS = [
    "general_1", "general_2", "general_3", "code_1", "code_2",
    "summarization", "instruction_1", "instruction_2", "long_ctx_1", "long_ctx_2",
]

MAX_NEW_TOKENS = 128
NUM_WARMUP = 3
NUM_TRIALS = 30
BASELINE_TPS = 26.42  # from Phase 1, for speedup calculation

def run_tree_benchmark(tree_root, name):
    results = []
    # warmup
    for _ in range(NUM_WARMUP):
        tree_spec_decode(PROMPTS[0], tree_root, max_new_tokens=16)

    torch.cuda.reset_peak_memory_stats()
    _prof_reset()

    for trial in range(NUM_TRIALS):
        pidx = trial % len(PROMPTS)
        prompt = PROMPTS[pidx]

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out_ids, n_iter, n_accept = tree_spec_decode(prompt, tree_root, MAX_NEW_TOKENS)
        torch.cuda.synchronize()
        wall = time.perf_counter() - t0

        prompt_len = tokenizer(prompt, return_tensors="pt").input_ids.shape[1]
        gen_tokens = out_ids.shape[1] - prompt_len

        results.append({
            "method": name,
            "prompt_idx": pidx,
            "gen_tokens": gen_tokens,
            "wall_sec": round(wall, 5),
            "tok_per_sec": round(gen_tokens / wall, 2) if wall > 0 else 0,
            "n_iter": n_iter,
            "n_accepted": n_accept,
            "tokens_per_iter": round(gen_tokens / n_iter, 2) if n_iter > 0 else 0,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6, 1),
        })
        if trial % 10 == 0:
            print(f"  trial {trial}: {results[-1]['tok_per_sec']:.2f} tok/s, {results[-1]['tokens_per_iter']} tok/iter")

    # Snapshot the profiler state at end of benchmark
    prof_snapshot = {
        "draft_prefix_ms_per_iter": (_PROF["draft_prefix_ms_sum"] / max(_PROF["iter_count"], 1)),
        "draft_extend_ms_per_iter": (_PROF["draft_extend_ms_sum"] / max(_PROF["iter_count"], 1)),
        "draft_extend_ms_per_call": (_PROF["draft_extend_ms_sum"] / max(_PROF["draft_extend_calls"], 1)),
        "draft_extend_calls_per_iter": (_PROF["draft_extend_calls"] / max(_PROF["iter_count"], 1)),
        "verify_ms_per_iter": (_PROF["verify_ms_sum"] / max(_PROF["iter_count"], 1)),
        "iter_count": _PROF["iter_count"],
    }
    return results, prof_snapshot


def summarize(results):
    tps = [r["tok_per_sec"] for r in results]
    tpi = [r["tokens_per_iter"] for r in results]
    walls = [r["wall_sec"] for r in results]
    mean_tps = float(np.mean(tps))
    return {
        "method": results[0]["method"],
        "mean_tok_sec": round(mean_tps, 2),
        "median_tok_sec": round(float(np.median(tps)), 2),
        "std_tok_sec": round(float(np.std(tps)), 2),
        "mean_tokens_per_iter": round(float(np.mean(tpi)), 2),
        "p95_wall_sec": round(float(np.percentile(walls, 95)), 4),
        "speedup_vs_baseline": round(mean_tps / BASELINE_TPS, 3),
        "peak_gpu_mb": max(r["peak_gpu_mb"] for r in results),
        "n_trials": len(results),
    }

print("Benchmark harness ready (with profiler hooks).")


Benchmark harness ready (with profiler hooks).


In [13]:
# ============================================================
# Cell F (updated): Run benchmarks with profiler capture
# ============================================================
ALL_RESULTS = []
ALL_SUMMARIES = []
PROF_SNAPSHOTS = {}

print("=" * 60)
print(f"BENCHMARK: tree (n={SECONDARY[0]}, d={SECONDARY[1]}) — sanity check")
print(f"DP-predicted speedup: {predicted_speedup(*SECONDARY):.3f}x")
print("=" * 60)

tree_small = BENCHMARK_TREES[SECONDARY]
res_small, prof_small = run_tree_benchmark(tree_small, f"tree_n{SECONDARY[0]}_d{SECONDARY[1]}")
ALL_RESULTS.extend(res_small)
sum_small = summarize(res_small)
sum_small["predicted_speedup"] = round(predicted_speedup(*SECONDARY), 3)
sum_small["tree_config"] = f"n={SECONDARY[0]}, d={SECONDARY[1]}"
ALL_SUMMARIES.append(sum_small)
PROF_SNAPSHOTS[f"n{SECONDARY[0]}_d{SECONDARY[1]}"] = prof_small
print(f"\nResult: {sum_small['mean_tok_sec']:.2f} tok/s  |  "
      f"speedup = {sum_small['speedup_vs_baseline']:.3f}x  |  "
      f"predicted = {sum_small['predicted_speedup']:.3f}x")
print(f"  Draft prefix/iter: {prof_small['draft_prefix_ms_per_iter']:.2f} ms | "
      f"extends/iter: {prof_small['draft_extend_calls_per_iter']:.2f} x "
      f"{prof_small['draft_extend_ms_per_call']:.2f} ms "
      f"= {prof_small['draft_extend_ms_per_iter']:.2f} ms")
print(f"  Verify/iter: {prof_small['verify_ms_per_iter']:.2f} ms")

# Save after each experiment in case of crash
with open("tree_results_partial.json", "w") as f:
    json.dump({"results": ALL_RESULTS, "summaries": ALL_SUMMARIES, "prof": PROF_SNAPSHOTS}, f, indent=2)

print("\n" + "=" * 60)
print(f"BENCHMARK: tree (n={PRIMARY[0]}, d={PRIMARY[1]}) — DP sweet spot")
print(f"DP-predicted speedup: {predicted_speedup(*PRIMARY):.3f}x")
print("=" * 60)

tree_primary = BENCHMARK_TREES[PRIMARY]
res_primary, prof_primary = run_tree_benchmark(tree_primary, f"tree_n{PRIMARY[0]}_d{PRIMARY[1]}")
ALL_RESULTS.extend(res_primary)
sum_primary = summarize(res_primary)
sum_primary["predicted_speedup"] = round(predicted_speedup(*PRIMARY), 3)
sum_primary["tree_config"] = f"n={PRIMARY[0]}, d={PRIMARY[1]}"
ALL_SUMMARIES.append(sum_primary)
PROF_SNAPSHOTS[f"n{PRIMARY[0]}_d{PRIMARY[1]}"] = prof_primary
print(f"\nResult: {sum_primary['mean_tok_sec']:.2f} tok/s  |  "
      f"speedup = {sum_primary['speedup_vs_baseline']:.3f}x  |  "
      f"predicted = {sum_primary['predicted_speedup']:.3f}x")
print(f"  Draft prefix/iter: {prof_primary['draft_prefix_ms_per_iter']:.2f} ms | "
      f"extends/iter: {prof_primary['draft_extend_calls_per_iter']:.2f} x "
      f"{prof_primary['draft_extend_ms_per_call']:.2f} ms "
      f"= {prof_primary['draft_extend_ms_per_iter']:.2f} ms")
print(f"  Verify/iter: {prof_primary['verify_ms_per_iter']:.2f} ms")


BENCHMARK: tree (n=8, d=3) — sanity check
DP-predicted speedup: 1.312x
  trial 0: 22.72 tok/s, 2.91 tok/iter
  trial 10: 20.34 tok/s, 2.91 tok/iter
  trial 20: 22.32 tok/s, 2.91 tok/iter

Result: 20.38 tok/s  |  speedup = 0.771x  |  predicted = 1.312x
  Draft prefix/iter: 30.85 ms | extends/iter: 2.00 x 22.16 ms = 44.31 ms
  Verify/iter: 64.46 ms

BENCHMARK: tree (n=16, d=3) — DP sweet spot
DP-predicted speedup: 1.682x
  trial 0: 8.90 tok/s, 2.91 tok/iter
  trial 10: 15.65 tok/s, 2.91 tok/iter
  trial 20: 14.92 tok/s, 2.91 tok/iter

Result: 14.71 tok/s  |  speedup = 0.557x  |  predicted = 1.682x
  Draft prefix/iter: 32.93 ms | extends/iter: 4.00 x 23.80 ms = 95.20 ms
  Verify/iter: 68.94 ms


In [14]:
# ============================================================
# Cell G: Summary and gap analysis
# ============================================================
print("=" * 70)
print("TREE SPECULATIVE DECODING RESULTS")
print("=" * 70)
print(f"{'Config':<16} {'Tok/s':>9} {'Speedup':>9} {'Predicted':>11} {'Gap':>7} {'Tok/iter':>9} {'GPU MB':>8}")
print("-" * 78)
for s in ALL_SUMMARIES:
    gap = s["speedup_vs_baseline"] - s["predicted_speedup"]
    print(f"{s['tree_config']:<16} {s['mean_tok_sec']:>9.2f} "
          f"{s['speedup_vs_baseline']:>8.3f}x {s['predicted_speedup']:>10.3f}x "
          f"{gap:>+6.3f} {s['mean_tokens_per_iter']:>9.2f} {s['peak_gpu_mb']:>8.0f}")

print("\n--- Cross-method comparison (vs Phase 1) ---")
print("Baseline AR           : 26.42 tok/s  | 1.000x (reference)")
print("Linear SD (k=5)       : 25.83 tok/s  | 0.977x")
print("PLD (n=5)             : 34.77 tok/s  | 1.316x")
for s in ALL_SUMMARIES:
    print(f"Tree {s['tree_config']:<12}  : {s['mean_tok_sec']:5.2f} tok/s  | {s['speedup_vs_baseline']:.3f}x")

print("\n--- Gap analysis ---")
print("Sources of measured < predicted in tree results:")
print(" 1. t(n) model is an extrapolation; real T4 verify cost may grow faster")
print(" 2. Csys for tree > Csys for linear (masked attention has extra ops)")
print(" 3. Draft tree traversal does k sequential draft passes per iteration")
print(" 4. Positional acceptance assumption may not hold perfectly in practice")

TREE SPECULATIVE DECODING RESULTS
Config               Tok/s   Speedup   Predicted     Gap  Tok/iter   GPU MB
------------------------------------------------------------------------------
n=8, d=3             20.38    0.771x      1.312x -0.541      2.88     9042
n=16, d=3            14.71    0.557x      1.682x -1.125      2.95     9014

--- Cross-method comparison (vs Phase 1) ---
Baseline AR           : 26.42 tok/s  | 1.000x (reference)
Linear SD (k=5)       : 25.83 tok/s  | 0.977x
PLD (n=5)             : 34.77 tok/s  | 1.316x
Tree n=8, d=3      : 20.38 tok/s  | 0.771x
Tree n=16, d=3     : 14.71 tok/s  | 0.557x

--- Gap analysis ---
Sources of measured < predicted in tree results:
 1. t(n) model is an extrapolation; real T4 verify cost may grow faster
 2. Csys for tree > Csys for linear (masked attention has extra ops)
 3. Draft tree traversal does k sequential draft passes per iteration
 4. Positional acceptance assumption may not hold perfectly in practice


In [15]:
# ============================================================
# Cell H: Save everything for the report
# ============================================================
output = {
    "target_model": TARGET_MODEL,
    "draft_model": DRAFT_MODEL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "n_trials": NUM_TRIALS,
    "baseline_tps": BASELINE_TPS,
    "cost_ratio_c": c_t4,
    "acceptance_vector": ACC_VECTOR,
    "configs": [PRIMARY, SECONDARY],
    "raw_results": ALL_RESULTS,
    "summaries": ALL_SUMMARIES,
    "profiler_snapshots": PROF_SNAPSHOTS,
    "cd_probe_results": probe_results if "probe_results" in dir() else None,
    "dp_prediction_method": "Sequoia Alg 1 (Appendix F.1.1) with measured acceptance vector",
    "implementation_notes": {
        "draft_kv_cache": "persistent prefix cache, O(1-token) extends per interior node",
        "cache_fork_policy": "fork when >1 interior child remains; last child consumes in place",
        "verify": "single target forward with tree attention mask + per-node position_ids",
    },
    "sources_of_gap": [
        "t(n) extrapolation error (T4 verify cost vs. predicted)",
        "tree attention mask construction overhead (Python-side)",
        "cache fork deep-copy cost (small but nonzero)",
        "positional acceptance assumption vs. real context-dependent acceptance",
    ],
}

with open("phase2b_tree_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved phase2b_tree_results.json")
print("\nDownload this file for the report.")


Saved phase2b_tree_results.json

Download this file for the report.


## Summary

This notebook implemented minimal Sequoia-compliant tree-structured speculative decoding:

1. **Sequoia DP optimizer** ported from Appendix F.1.1, fed with Phase-2A-measured acceptance vector, produced optimal tree structures.
2. **Tree runtime** implemented draft population (top-k at each parent), target verification with ancestors-only attention mask, and cover-property longest-path acceptance.
3. **Correctness verified**: greedy tree output matches greedy baseline on the first 20 tokens.
4. **Benchmarked** DP-optimal tree and a smaller sanity-check config, 30 trials × 10 prompts each (matching Phase 1 methodology).
5. **Gap analysis** between DP-predicted and empirically measured speedup identifies concrete sources of the gap for the report.

Output: `phase2b_tree_results.json` — downloadable for the report.

**Update (revised Phase 2B):** `draft_populate_tree` was rewritten to maintain a persistent draft KV cache across tree-depth levels. Each non-root interior node extends its parent-branch cache by exactly one token (the parent's tree token) rather than reprocessing the full prefix. Cache forks (per-layer K/V tensor clones) are made only when multiple interior siblings remain; the last interior child consumes the parent cache in place. This matches Sequoia's cost model `S = G(n,d) / (t(n) + d·c)`, which assumes O(d) incremental draft forwards per iteration, not O(|interior|) full-prefix forwards. Correctness is preserved (target-side verification is unchanged).


**Update 2 (Cd/Ct probe):** To close the loop on the framework-amortization hypothesis from checkpoint 2, a new probe cell measures per-call 1-token incremental decode cost for both draft and target across cache lengths {30, 64, 128, 256}, plus standalone autoregressive Cd/Ct with persistent cache, plus the cost of `copy.deepcopy(cache)`. This lets the report's 'framework amortization' section either (a) confirm bandwidth-bound per-call Cd is ~constant in L, invalidating the 10x amortization claim, or (b) reveal that HF's assisted generation does something beyond KV persistence.
